# T9 — Prototipo / Dashboard de Demostración

**Tesis:** "Diseño e Implementación de un Sistema de Recomendación Laboral  
para Optimizar el Matching entre Candidatos y Ofertas de Empleo"  
**Alumno:** Cicconi, Carlos Alberto  
**Maestría:** Explotación de Datos y Gestión del Conocimiento — Universidad Austral  
**Período:** 17/03/2026 – 31/03/2026

---

## Objetivo de esta etapa

Desarrollar el prototipo funcional del sistema que integra todos los módulos  
construidos en T2–T8 en una interfaz interactiva demostrable para la tesis.  
El entregable principal es `app.py`, una aplicación **Streamlit** que permite:  

- Ingresar un CV (PDF) y obtener recomendaciones en tiempo real
- Ajustar interactivamente la ponderación del motor híbrido (SBERT / TF-IDF)
- Visualizar el benchmark comparativo de los 5 sistemas evaluados en T7
- Explorar el corpus con gráficos interactivos y mapa vectorial 2D

## Entregables

| Archivo | Descripción |
|---|---|
| `app.py` | Aplicación Streamlit completa (664 líneas) |
| `T9_Prototipo_Dashboard.ipynb` | Este notebook: documentación, validación y capturas |

## Inputs requeridos (en el mismo directorio que `app.py`)

| Archivo | Origen | Obligatorio |
|---|---|---|
| `ranking_final_optimizado.csv` | T8 | ✅ (o cualquier ranking de T6/T7) |
| `embeddings_ofertas.npy` | T5 | Opcional (habilita PCA y scoring en vivo) |
| `embedding_cv.npy` | T5 | Opcional |
| `tfidf_vectorizer.pkl` | T4 | Opcional (habilita scoring de CV nuevo) |
| `benchmark_metricas.json` | T7 | Opcional (habilita pestaña de benchmark) |
| `config_optima.json` | T8 | Opcional (pre-carga w_opt en el slider) |
| `tuning_resultados.csv` | T8 | Opcional (habilita curva de tuning) |
| `CarlosA_Cicconi_CV.pdf` | — | Opcional (CV por defecto del candidato) |


## 0. Verificación del entorno

In [1]:
# Verificar que Streamlit esté instalado
# Si no está: pip install streamlit plotly
import subprocess, sys

paquetes = ['streamlit', 'plotly']
for pkg in paquetes:
    try:
        __import__(pkg)
        import importlib.metadata as im
        version = im.version(pkg)
        print(f'✅ {pkg:<15} {version}')
    except ImportError:
        print(f'❌ {pkg:<15} NO instalado — ejecutar: pip install {pkg}')

# Verificar artefactos disponibles
from pathlib import Path
print()
print('Artefactos disponibles:')
artefactos = [
    ('ranking_final_optimizado.csv', 'ranking T8 — OBLIGATORIO'),
    ('ranking_final_combinado.csv',  'ranking T6 — fallback'),
    ('ofertas_preprocesadas.csv',    'corpus T4'),
    ('embeddings_ofertas.npy',       'vectores SBERT T5'),
    ('embedding_cv.npy',             'vector CV T5'),
    ('tfidf_vectorizer.pkl',         'vectorizador T4'),
    ('benchmark_metricas.json',      'benchmark T7'),
    ('config_optima.json',           'config óptima T8'),
    ('tuning_resultados.csv',        'tuning T8'),
    ('CarlosA_Cicconi_CV.pdf',       'CV candidato'),
]
for fname, desc in artefactos:
    p = Path(fname)
    if p.exists():
        size = p.stat().st_size / 1024
        print(f'  ✅ {fname:<40} {size:>8.0f} KB  — {desc}')
    else:
        print(f'  ❌ {fname:<40} NO encontrado — {desc}')


✅ streamlit       1.44.1
✅ plotly          6.0.1

Artefactos disponibles:
  ✅ ranking_final_optimizado.csv                  310 KB  — ranking T8 — OBLIGATORIO
  ✅ ranking_final_combinado.csv                   333 KB  — ranking T6 — fallback
  ✅ ofertas_preprocesadas.csv                    5830 KB  — corpus T4
  ✅ embeddings_ofertas.npy                       1919 KB  — vectores SBERT T5
  ✅ embedding_cv.npy                                2 KB  — vector CV T5
  ✅ tfidf_vectorizer.pkl                          634 KB  — vectorizador T4
  ✅ benchmark_metricas.json                         3 KB  — benchmark T7
  ✅ config_optima.json                              1 KB  — config óptima T8
  ✅ tuning_resultados.csv                           1 KB  — tuning T8
  ✅ CarlosA_Cicconi_CV.pdf                        146 KB  — CV candidato


## 1. Arquitectura del prototipo

### Estructura de `app.py`

```
app.py
│
├── Configuración de página (st.set_page_config)
├── Carga de datos (@st.cache_data / @st.cache_resource)
│     ├── cargar_corpus()        → DataFrame principal
│     ├── cargar_embeddings()    → embeddings_ofertas.npy + embedding_cv.npy
│     ├── cargar_tfidf()         → tfidf_vectorizer.pkl
│     ├── cargar_metricas()      → benchmark_metricas.json
│     ├── cargar_config()        → config_optima.json
│     └── cargar_tuning()        → tuning_resultados.csv
│
├── Sidebar
│     ├── Selector de CV (por defecto / subir PDF)
│     ├── Slider ponderación w_embed (pre-cargado con w_opt de T8)
│     └── Filtros Top-N
│
└── Tabs
      ├── Tab 1: Recomendaciones
      │     ├── Filtros (keyword / fuente / modalidad)
      │     ├── Métricas resumen (4 KPIs)
      │     ├── Tarjetas expandibles con gauge de score
      │     └── Exportar CSV
      ├── Tab 2: Benchmark y Métricas
      │     ├── Tabla comparativa 5 sistemas (resaltado del máximo)
      │     ├── Gráficos Precision@K y nDCG@K (Plotly)
      │     └── Curva de tuning ponderación (T8)
      └── Tab 3: Exploración del Corpus
            ├── KPIs del corpus
            ├── Distribución por fuente / keyword / modalidad / idioma
            ├── Mapa vectorial PCA 2D (con ★ CV del candidato)
            └── Heatmap fuente × keyword
```

### Decisiones de diseño

| Decisión | Justificación |
|---|---|
| **Streamlit** sobre Flask/FastAPI | Desarrollo en Python puro, sin HTML/JS, ideal para prototipo académico |
| **Plotly** sobre matplotlib | Gráficos interactivos (zoom, hover, filtros) sin código adicional |
| **@st.cache_data** | Los artefactos pesados (embeddings, corpus) se cargan una sola vez |
| **Tabs** en lugar de páginas separadas | Navegación fluida sin recargar el estado de la app |
| **Gauge por oferta** | Visualización intuitiva del score para audiencia no técnica |
| **Fallback en cascada** | La app funciona aunque falten artefactos (usa lo disponible) |


## 2. Instrucciones de ejecución

### Instalación de dependencias adicionales

```bash
# Activar el entorno del proyecto
source venv_tesis/bin/activate

# Instalar Streamlit y Plotly (si no están)
pip install streamlit plotly
```

### Ejecución del dashboard

```bash
# Desde el directorio donde están los artefactos y app.py
streamlit run app.py
```

El dashboard se abre automáticamente en `http://localhost:8501`

### Ejecución en modo headless (servidor remoto)

```bash
streamlit run app.py --server.headless true --server.port 8501
```


In [2]:
# Validar que app.py existe y tiene el contenido correcto
from pathlib import Path

app_path = _DASHBOARD / 'app.py'
if app_path.exists():
    contenido = app_path.read_text(encoding='utf-8')
    lineas    = contenido.count(chr(10))
    funciones = [l.strip() for l in contenido.split(chr(10))
                 if l.strip().startswith('def ') or l.strip().startswith('class ')]
    print(f'✅ app.py encontrado')
    print(f'   Líneas       : {lineas:,}')
    print(f'   Funciones/clases:')
    for f in funciones:
        print(f'     {f}')
else:
    print('❌ app.py no encontrado en el directorio actual.')
    print('   Copiá app.py al mismo directorio que los artefactos.')


✅ app.py encontrado
   Líneas       : 664
   Funciones/clases:
     def cargar_corpus() -> pd.DataFrame:
     def cargar_embeddings():
     def cargar_tfidf():
     def cargar_metricas():
     def cargar_config():
     def cargar_tuning():
     def normalizar_mm(serie: pd.Series) -> pd.Series:
     def leer_cv_pdf(archivo) -> str:
     def calcular_score_hibrido(df: pd.DataFrame,
     def calcular_score_cv_nuevo(cv_texto: str, df: pd.DataFrame,
     def badge_fuente(fuente: str) -> str:
     def badge_modalidad(modalidad: str) -> str:
     def render_sidebar(config: dict) -> tuple:
     def pagina_recomendaciones(df: pd.DataFrame, emb_of, emb_cv,
     def pagina_benchmark(metricas: dict, df_tuning: pd.DataFrame):
     def highlight_max(s):
     def pagina_corpus(df: pd.DataFrame, emb_of, emb_cv):
     def main():


## 3. Validación técnica del prototipo

In [3]:
# Smoke test: importar y ejecutar las funciones principales sin Streamlit
import sys, types

# Crear mock de streamlit para poder importar app.py sin levantar el servidor
class MockSt:
    def __getattr__(self, name):
        return lambda *a, **kw: None
    def columns(self, n):
        if isinstance(n, int):
            return [self] * n
        return [self] * len(n)
    def tabs(self, names):
        return [self] * len(names)
    def __enter__(self): return self
    def __exit__(self, *a): pass
    cache_data     = lambda self, *a, **kw: (lambda f: f)
    cache_resource = lambda self, *a, **kw: (lambda f: f)
    set_page_config = lambda self, *a, **kw: None
    stop = lambda self: None

sys.modules['streamlit'] = MockSt()

from pathlib import Path
import importlib.util

if (_DASHBOARD / 'app.py').exists():
    spec   = importlib.util.spec_from_file_location('app', _DASHBOARD / 'app.py')
    modulo = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(modulo)
        print('✅ app.py importa correctamente (sin errores de sintaxis)')

        # Verificar funciones clave
        funciones_requeridas = [
            'cargar_corpus', 'cargar_embeddings', 'cargar_tfidf',
            'cargar_metricas', 'normalizar_mm', 'calcular_score_hibrido',
            'pagina_recomendaciones', 'pagina_benchmark', 'pagina_corpus',
        ]
        for fn in funciones_requeridas:
            exists = hasattr(modulo, fn)
            print(f'  {"✅" if exists else "❌"} {fn}')
    except Exception as e:
        print(f'❌ Error al importar app.py: {e}')
else:
    print('❌ app.py no encontrado')


✅ app.py importa correctamente (sin errores de sintaxis)
  ✅ cargar_corpus
  ✅ cargar_embeddings
  ✅ cargar_tfidf
  ✅ cargar_metricas
  ✅ normalizar_mm
  ✅ calcular_score_hibrido
  ✅ pagina_recomendaciones
  ✅ pagina_benchmark
  ✅ pagina_corpus


In [4]:
# Validar que las funciones de scoring producen resultados coherentes
import numpy as np
import pandas as pd
from pathlib import Path

print('=== VALIDACIÓN DE FUNCIONES DE SCORING ===')
print()

# Función de normalización
def normalizar_mm(serie):
    mn, mx = serie.min(), serie.max()
    return (serie - mn) / (mx - mn) if mx > mn else serie * 0

# Test con datos sintéticos
np.random.seed(42)
n = 100
df_test = pd.DataFrame({
    'id':              range(n),
    'titulo':          [f'Oferta {i}' for i in range(n)],
    'similitud_embed': np.random.uniform(0.1, 0.9, n),
    'similitud_tfidf': np.random.uniform(0.0, 0.4, n),
})

# Test normalización
em_n = normalizar_mm(df_test['similitud_embed'])
tf_n = normalizar_mm(df_test['similitud_tfidf'])
assert em_n.min() >= 0 and em_n.max() <= 1, 'Error: normalización fuera de [0,1]'
print('✅ normalizar_mm: rango correcto [0, 1]')

# Test score híbrido
for w in [0.0, 0.5, 0.7, 1.0]:
    score = w * em_n + (1 - w) * tf_n
    assert score.min() >= 0 and score.max() <= 1, f'Score fuera de rango para w={w}'
print('✅ score híbrido: valores en [0, 1] para w ∈ {0.0, 0.5, 0.7, 1.0}')

# Test ranking
score_final = 0.7 * em_n + 0.3 * tf_n
ranking = df_test.assign(score=score_final).sort_values('score', ascending=False)
assert ranking['score'].iloc[0] >= ranking['score'].iloc[-1], 'Error: ranking no ordenado'
print('✅ ranking: orden descendente correcto')

print()
print(f'Score promedio (w=0.7): {score_final.mean():.4f}')
print(f'Score máximo  (w=0.7): {score_final.max():.4f}')
print(f'Top-3 sintético: {ranking["id"].head(3).tolist()}')

# ── Rutas del proyecto (resueltas desde la ubicación del notebook) ────────────
from pathlib import Path
_NB   = Path(globals().get('__vsc_ipynb_file__', Path.cwd())).resolve()
_ROOT = _NB.parent if _NB.name != 'notebooks' else _NB.parent
if _ROOT.name == 'notebooks': _ROOT = _ROOT.parent
_PROC      = _ROOT / 'data'    / 'processed'
_EMBEDDINGS= _ROOT / 'data'    / 'embeddings'
_MODELS    = _ROOT / 'outputs' / 'models'
_RANKINGS  = _ROOT / 'outputs' / 'rankings'
_REPORTS   = _ROOT / 'outputs' / 'reports'
_FIGS      = _ROOT / 'outputs' / 'figures'
_DASHBOARD = _ROOT / 'dashboard'
print(f'ROOT       : {_ROOT}')
print(f'PROC       : {_PROC}')
print(f'EMBEDDINGS : {_EMBEDDINGS}')
print(f'MODELS     : {_MODELS}')
print(f'RANKINGS   : {_RANKINGS}')
print(f'REPORTS    : {_REPORTS}')
print(f'DASHBOARD  : {_DASHBOARD}')


=== VALIDACIÓN DE FUNCIONES DE SCORING ===

✅ normalizar_mm: rango correcto [0, 1]
✅ score híbrido: valores en [0, 1] para w ∈ {0.0, 0.5, 0.7, 1.0}
✅ ranking: orden descendente correcto

Score promedio (w=0.7): 0.4819
Score máximo  (w=0.7): 0.9717
Top-3 sintético: [34, 50, 69]


## 4. Demo con datos reales

Simulación del flujo completo del dashboard sobre los artefactos reales,  
sin levantar el servidor Streamlit. Permite verificar el output antes de ejecutar la app.


In [5]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# ── Cargar corpus ─────────────────────────────────────────────────────────────
# Estrategia en cascada:
#   1. Archivos con descripcion completa (preferidos para el filtro por largo)
#   2. Archivos de ranking (no tienen descripcion — se carga corpus separado)

ARCHIVOS_CON_DESC = [
    _PROC     / 'ofertas_preprocesadas.csv',
    _PROC     / 'ofertas_con_descripcion.csv',
]
ARCHIVOS_RANKING = [
    _RANKINGS / 'ranking_final_optimizado.csv',
    _RANKINGS / 'ranking_final_combinado.csv',
]

df = None

# Intentar primero archivos con descripcion
for fpath in ARCHIVOS_CON_DESC:
    if fpath.exists():
        fname = fpath.name
        df = pd.read_csv(fpath, low_memory=False)
        if 'descripcion' in df.columns:
            df = df[df['descripcion'].fillna('').str.len() > 50].reset_index(drop=True)
        print(f'✅ Corpus (con descripcion): {fname} — {len(df):,} filas')
        break

# Si no hay archivo con descripcion, usar el ranking y enriquecer si es posible
if df is None:
    for fname in ARCHIVOS_RANKING:
        if Path(fname).exists():
            df = pd.read_csv(fname, low_memory=False)
            print(f'✅ Corpus (ranking): {fname} — {len(df):,} filas')
            # Intentar enriquecer con descripcion desde otro archivo
            for desc_fpath in ARCHIVOS_CON_DESC:
                if desc_fpath.exists():
                    df_desc = pd.read_csv(desc_fpath, low_memory=False,
                                          usecols=lambda c: c in
                                          ['id_consolidado','descripcion'])
                    if 'id_consolidado' in df.columns and 'id_consolidado' in df_desc.columns:
                        df = df.merge(df_desc, on='id_consolidado', how='left')
                        print(f'   ℹ️  Descripción enriquecida desde {desc_fname}')
                    break
            if 'descripcion' not in df.columns:
                df['descripcion'] = ''
                print('   ⚠️  Sin columna descripcion — se usará cadena vacía')
            break

if df is None:
    print('❌ No se encontró ningún archivo de corpus.')
else:
    print(f'   Columnas disponibles: {df.columns.tolist()}')


✅ Corpus (con descripcion): ofertas_preprocesadas.csv — 1,279 filas
   Columnas disponibles: ['id_consolidado', 'job_id', 'fuente', 'titulo', 'empresa', 'ubicacion', 'descripcion', 'fecha_pub_dt', 'url', 'keyword', 'nivel', 'tipo_empleo', 'modalidad', 'salario', 'fecha_scrap', 'idioma', 'texto_procesado', 'similitud_tfidf']


In [6]:
# ── Calcular score y mostrar Top-10 ──────────────────────────────────────────
if df is not None:
    def normalizar_mm(serie):
        mn, mx = serie.min(), serie.max()
        return (serie - mn) / (mx - mn) if mx > mn else serie * 0

    # Leer w_opt desde config si existe
    w_opt = 0.70
    if (_REPORTS / 'config_optima.json').exists():
        with open(_REPORTS / 'config_optima.json') as f:
            cfg = json.load(f)
        w_opt = cfg.get('tuning_A', {}).get('w_embed', 0.70)
        print(f'w_opt desde config_optima.json: {w_opt}')

    # Calcular score
    if 'score_optimizado' in df.columns:
        df['score_demo'] = df['score_optimizado']
        print('Score: desde score_optimizado (T8)')
    elif 'score_final' in df.columns:
        df['score_demo'] = df['score_final']
        print('Score: desde score_final (T6)')
    elif 'similitud_embed' in df.columns and 'similitud_tfidf' in df.columns:
        em_n = normalizar_mm(df['similitud_embed'].fillna(0))
        tf_n = normalizar_mm(df['similitud_tfidf'].fillna(0))
        df['score_demo'] = w_opt * em_n + (1 - w_opt) * tf_n
        print(f'Score: híbrido recalculado (w={w_opt})')
    elif 'similitud_embed' in df.columns:
        df['score_demo'] = normalizar_mm(df['similitud_embed'].fillna(0))
        print('Score: solo SBERT')
    else:
        df['score_demo'] = 0
        print('⚠️  Sin similitudes disponibles')

    ranking = df.sort_values('score_demo', ascending=False).reset_index(drop=True)
    ranking['rank'] = ranking.index + 1

    print()
    print('=== TOP 10 — Demo del dashboard ===')
    cols_show = ['rank','score_demo','titulo','empresa','keyword','fuente','modalidad']
    cols_show = [c for c in cols_show if c in ranking.columns]
    top10 = ranking[cols_show].head(10).copy()
    top10['score_demo'] = top10['score_demo'].round(4)
    print(top10.to_string(index=False))


w_opt desde config_optima.json: 0.7
⚠️  Sin similitudes disponibles

=== TOP 10 — Demo del dashboard ===
 rank  score_demo                                                                titulo               empresa                keyword       fuente modalidad
    1           0 Desarrollo de Sitio Web Inmobiliario con Integración de Reseñas y Crm      Freelancer F. M.         Data Scientist      workana    Remoto
    2           0                     Manager, Analytics and Optimization - Client Care                CHANEL Continuous Improvement     linkedin       NaN
    3           0                                               People & Talent Partner             Ticketure Continuous Improvement     linkedin       NaN
    4           0                              Business Strategy-Continuous Improvement             Microsoft Continuous Improvement     linkedin       NaN
    5           0                               Digital Manufacturing Engineer (Remote) Capgemini Engineering   Ing

In [7]:
# ── Estadísticas del corpus para la pestaña Exploración ──────────────────────
if df is not None:
    print('=== ESTADÍSTICAS DEL CORPUS ===')
    print(f'  Total ofertas      : {len(df):,}')

    if 'fuente' in df.columns:
        print(f'  Fuentes            : {df["fuente"].nunique()}')
        print(df['fuente'].value_counts().to_string())
        print()

    if 'keyword' in df.columns:
        print(f'  Keywords           : {df["keyword"].nunique()}')
        print(df['keyword'].value_counts().to_string())
        print()

    if 'modalidad' in df.columns:
        print('  Modalidad:')
        print(df['modalidad'].value_counts(dropna=False).to_string())
        print()

    if 'idioma' in df.columns:
        print('  Idioma:')
        print(df['idioma'].value_counts().to_string())


=== ESTADÍSTICAS DEL CORPUS ===
  Total ofertas      : 1,279
  Fuentes            : 5
fuente
opcionempleo    761
linkedin        373
bumeran          97
workana          37
jobleads         11

  Keywords           : 11
keyword
Data Scientist            159
Control de Gestión        154
Compras / Procurement     142
Excel / VBA               129
BI / Analytics            122
Ingeniero Industrial      111
Supply Chain              110
Continuous Improvement     98
Python Developer           89
Gerente de Operaciones     88
Finanzas / CAPEX           77

  Modalidad:
modalidad
NaN           1145
Presencial      69
Remoto          40
Híbrido         25

  Idioma:
idioma
es    827
en    452


## 5. Visualizaciones estáticas (para la tesis)

Reproducción de los gráficos principales del dashboard en formato estático  
para incluir en el documento de la tesis.


In [8]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 150, 'axes.spines.top': False,
                     'axes.spines.right': False})

# ── Fig 1: Benchmark comparativo ─────────────────────────────────────────────
if (_REPORTS / 'benchmark_metricas.json').exists():
    with open(_REPORTS / 'benchmark_metricas.json') as f:
        bench = json.load(f)

    sistemas = bench.get('sistemas', {})
    if sistemas:
        Ks     = [5, 10, 20]
        names  = [n.split(' — ')[1] if ' — ' in n else n for n in sistemas]
        colores = sns.color_palette('tab10', len(sistemas))
        marcad  = ['o','s','D','^','*']

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))

        for i, (nombre, vals) in enumerate(sistemas.items()):
            nc = nombre.split(' — ')[1] if ' — ' in nombre else nombre
            lw = 2.5 if 'SBERT' in nombre and 'Híbrido' in nombre else 1.5
            ls = '-'  if 'SBERT' in nombre and 'Híbrido' in nombre else '--'
            axes[0].plot(Ks, [vals.get(f'P@{k}',0) for k in Ks],
                         marker=marcad[i], label=nc, color=colores[i], lw=lw, ls=ls)
            axes[1].plot(Ks, [vals.get(f'nDCG@{k}',0) for k in Ks],
                         marker=marcad[i], label=nc, color=colores[i], lw=lw, ls=ls)

        for ax, titulo in zip(axes, ['Precision@K', 'nDCG@K']):
            ax.set_xlabel('K'); ax.set_ylabel(titulo)
            ax.set_title(titulo); ax.legend(fontsize=8); ax.set_xticks(Ks)

        plt.suptitle('Benchmark comparativo — 5 sistemas de matching', fontsize=12)
        plt.tight_layout()
        plt.savefig(_FIGS / 'fig_t9_01_benchmark.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('✅ fig_t9_01_benchmark.png')
else:
    print('ℹ️  benchmark_metricas.json no encontrado')


✅ fig_t9_01_benchmark.png


In [9]:
# ── Fig 2: Distribución del corpus ───────────────────────────────────────────
if df is not None and 'fuente' in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Por fuente
    cnt_f = df['fuente'].value_counts().sort_values()
    cnt_f.plot(kind='barh', ax=axes[0],
               color=sns.color_palette('muted', len(cnt_f)))
    axes[0].set_title('Ofertas por fuente')
    axes[0].set_xlabel('N ofertas')

    # Por keyword
    if 'keyword' in df.columns:
        cnt_k = df['keyword'].value_counts().sort_values()
        cnt_k.plot(kind='barh', ax=axes[1],
                   color=sns.color_palette('muted', len(cnt_k)))
        axes[1].set_title('Ofertas por keyword')
        axes[1].set_xlabel('N ofertas')

    # Por modalidad
    if 'modalidad' in df.columns:
        cnt_m = df['modalidad'].value_counts(dropna=False)
        cnt_m.index = cnt_m.index.fillna('Sin datos')
        cnt_m.plot(kind='bar', ax=axes[2],
                   color=sns.color_palette('muted', len(cnt_m)))
        axes[2].set_title('Ofertas por modalidad')
        axes[2].set_ylabel('N ofertas')
        axes[2].tick_params(axis='x', rotation=25)

    plt.suptitle('Exploración del corpus de ofertas laborales', fontsize=12)
    plt.tight_layout()
    plt.savefig(_FIGS / 'fig_t9_02_corpus.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ fig_t9_02_corpus.png')


✅ fig_t9_02_corpus.png


In [10]:
# ── Fig 3: Mapa vectorial PCA 2D (estático) ──────────────────────────────────
import numpy as np
from sklearn.decomposition import PCA

if (_EMBEDDINGS / 'embeddings_ofertas.npy').exists() and df is not None:
    emb_of = np.load(_EMBEDDINGS / 'embeddings_ofertas.npy')

    # Alinear tamaños (puede haber diferencia si el corpus fue filtrado)
    n_min = min(len(df), len(emb_of))
    emb_of = emb_of[:n_min]
    df_pca_src = df.iloc[:n_min].copy()

    if (_EMBEDDINGS / 'embedding_cv.npy').exists():
        emb_cv = np.load(_EMBEDDINGS / 'embedding_cv.npy')
        all_emb = np.vstack([emb_of, emb_cv.reshape(1,-1)])
    else:
        all_emb = emb_of
        emb_cv  = None

    pca    = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(all_emb)
    c_of   = coords[:len(emb_of)]
    c_cv   = coords[len(emb_of)] if emb_cv is not None else None

    fig, ax = plt.subplots(figsize=(10, 7))

    if 'keyword' in df_pca_src.columns:
        kws     = df_pca_src['keyword'].unique()
        paleta  = sns.color_palette('tab20', len(kws))
        kw_col  = {kw: c for kw, c in zip(kws, paleta)}
        for kw in kws:
            mask = df_pca_src['keyword'] == kw
            ax.scatter(c_of[mask,0], c_of[mask,1],
                       color=kw_col[kw], s=18, alpha=0.5, label=kw)
        ax.legend(bbox_to_anchor=(1.01,1), loc='upper left',
                  fontsize=7, frameon=True)
    else:
        ax.scatter(c_of[:,0], c_of[:,1], s=18, alpha=0.5, color='steelblue')

    if c_cv is not None:
        ax.scatter(c_cv[0], c_cv[1], s=300, color='black',
                   marker='*', zorder=10, label='CV candidato')
        ax.annotate('★ CV', (c_cv[0], c_cv[1]),
                    fontsize=10, color='black',
                    xytext=(8,8), textcoords='offset points')

    var_exp = pca.explained_variance_ratio_.sum() * 100
    ax.set_title(f'Espacio vectorial SBERT — PCA 2D\n'
                 f'Varianza explicada: {var_exp:.1f}%', fontsize=12)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    plt.tight_layout()
    plt.savefig(_FIGS / 'fig_t9_03_pca_vectorial.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ fig_t9_03_pca_vectorial.png')
else:
    print('ℹ️  embeddings_ofertas.npy no disponible — figura omitida')


✅ fig_t9_03_pca_vectorial.png


## 6. Resumen ejecutivo de T9

In [11]:
from pathlib import Path
from datetime import datetime

print('=' * 65)
print('  RESUMEN — T9: Prototipo / Dashboard de Demostración')
print('=' * 65)
print(f"  Fecha : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print()
print('  ENTREGABLES')
for p, desc in [
    (_DASHBOARD / 'app.py',                        'Aplicación Streamlit completa'),
    (Path('T9_Prototipo_Dashboard.ipynb'),          'Notebook de documentación y validación'),
]:
    fn = p.name
    if p.exists():
        print(f'    ✅ {fn:<45} {p.stat().st_size/1024:.0f} KB')
    else:
        print(f'    ❌ {fn:<45} NO encontrado')
print()
print('  ARTEFACTOS DE ENTRADA')
for p in [
    _RANKINGS   / 'ranking_final_optimizado.csv',
    _EMBEDDINGS / 'embeddings_ofertas.npy',
    _EMBEDDINGS / 'embedding_cv.npy',
    _MODELS     / 'tfidf_vectorizer.pkl',
    _REPORTS    / 'benchmark_metricas.json',
    _REPORTS    / 'config_optima.json',
]:
    fn = p.name
    estado = f'✅ {p.stat().st_size/1024:.0f} KB' if p.exists() else '❌ no encontrado'
    print(f'    {fn:<45} {estado}')
print()
print('  CARACTERÍSTICAS DEL DASHBOARD')
print('    - 3 pestañas: Recomendaciones / Benchmark / Corpus')
print('    - Slider interactivo de ponderación SBERT vs TF-IDF')
print('    - Soporte para subir CV propio (PDF)')
print('    - Tarjetas con gauge de score por oferta')
print('    - Exportación de resultados en CSV')
print('    - Mapa vectorial PCA 2D interactivo (Plotly)')
print('    - Gráficos benchmark con Precision@K y nDCG@K')
print()
print('  CÓMO EJECUTAR')
print('    streamlit run app.py')
print('    → http://localhost:8501')
print()
print('  PRÓXIMO PASO: T10 — Redacción Final y Revisión de la Tesis')
print('=' * 65)


  RESUMEN — T9: Prototipo / Dashboard de Demostración
  Fecha : 2026-04-27 16:08

  ENTREGABLES
    ✅ app.py                                        30 KB
    ✅ T9_Prototipo_Dashboard.ipynb                  30 KB

  ARTEFACTOS DE ENTRADA
    ranking_final_optimizado.csv                  ✅ 310 KB
    embeddings_ofertas.npy                        ✅ 1919 KB
    embedding_cv.npy                              ✅ 2 KB
    tfidf_vectorizer.pkl                          ✅ 634 KB
    benchmark_metricas.json                       ✅ 3 KB
    config_optima.json                            ✅ 1 KB

  CARACTERÍSTICAS DEL DASHBOARD
    - 3 pestañas: Recomendaciones / Benchmark / Corpus
    - Slider interactivo de ponderación SBERT vs TF-IDF
    - Soporte para subir CV propio (PDF)
    - Tarjetas con gauge de score por oferta
    - Exportación de resultados en CSV
    - Mapa vectorial PCA 2D interactivo (Plotly)
    - Gráficos benchmark con Precision@K y nDCG@K

  CÓMO EJECUTAR
    streamlit run app.py
    